# 🛡️ 保险营销内容智能审核系统 - Jupyter Notebook Demo

本 demo 直接在 Jupyter Notebook 中运行，无需网络穿透，完美适配云服务器环境。
## 🎯 功能特点
- ✅ 文本输入审核
- ✅ 实时结果展示
- ✅ 违规项可视化
- ✅ 多种审核模式切换
- ✅ 示例文案快速测试

In [2]:
# 导入必要的库
import sys
import os
from pathlib import Path

# 添加项目路径
sys.path.insert(0, '/mnt/workspace')

import json
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

# 导入项目模块
from aliyun_rag.bailian_client import BailianClient
from aliyun_rag.config import Settings
from aliyun_rag.knowledge_base import load_knowledge_base
from aliyun_rag.auditor import audit_marketing_text
from aliyun_rag.enhanced_auditor import enhanced_audit_marketing_text

print("✅ 导入成功！")

✅ 导入成功！


## 📦 1. 加载资源

In [3]:
# 加载配置和知识库
settings = Settings.from_env()
settings.validate()
client = BailianClient(settings)
kb = load_knowledge_base('kb')

print("✅ 资源加载成功！")
print(f"📚 知识库: {len(kb.chunks)} 个法规 chunks")

FileNotFoundError: Knowledge base files not found in kb. Run build-kb first.

## 📝 2. 文本审核 Demo

输入营销文案，选择审核模式，点击运行查看结果。

In [ ]:
# ==================== 用户配置区 ====================

# 📝 输入营销文案
marketing_text = "本保险保证年化收益8%，稳赚不赔。"

# 🔄 选择审核模式
use_enhanced = False  # True=增强模式, False=基础模式

# 📊 选择检索方式
use_hybrid = False  # True=混合检索, False=仅向量检索

# 🔢 检索相关条文数量
top_k = 6

# ==================== 执行审核 ====================
print("🔍 开始审核...")
print(f"📝 营销文案: {marketing_text}")
print(f"📊 审核模式: {'增强模式' if use_enhanced else '基础模式'}")
print(f"🔀 检索方式: {'混合检索' if use_hybrid else '向量检索'}")
print()

# 执行审核
if use_enhanced:
    result = enhanced_audit_marketing_text(
        marketing_text=marketing_text,
        kb=kb,
        client=client,
        top_k=top_k,
        enable_math_confidence=True,
    )
else:
    result = audit_marketing_text(
        marketing_text=marketing_text,
        kb=kb,
        client=client,
        top_k=top_k,
    )

# 显示结果
print("\n" + "="*70)
print("📊 审核结果")
print("="*70)
print()

# 是否合规
is_compliant = result.get('is_compliant', 'unknown')
status_icon = "✅" if is_compliant == "yes" else "❌"
print(f"{status_icon} 是否合规: {is_compliant.upper()}")
print()

# 置信度
if 'calculated_confidence' in result:
    print(f"📈 置信度分析:")
    print(f"   - LLM置信度: {result['llm_confidence']:.2%}")
    print(f"   - 计算置信度: {result['calculated_confidence']:.2%}")
    print(f"   - 融合置信度: {result['overall_confidence']:.2%}")
else:
    print(f"📈 整体置信度: {result.get('overall_confidence', 0):.2%}")
print()

# 总结
print(f"📝 总结: {result.get('summary', 'N/A')}")
print()

# 意图分析（增强模式）
if 'intent_analysis' in result:
    intent = result['intent_analysis']
    print("🎯 意图分析:")
    print(f"   - 主要意图: {intent.get('primary_intent', 'N/A')}")
    print(f"   - 检测到的风险: {', '.join(intent.get('detected_risks', []))}")
    print(f"   - 风险等级: {intent.get('risk_level', 'N/A')}")
    print()

# 违规项详情
violations = result.get('violations', [])
if violations:
    print("⚠️  违规项:")
    for i, v in enumerate(violations, 1):
        print(f"\n  [{i}] {v.get('type', 'N/A')}")
        print(f"      条文: {v.get('clause_id', 'N/A')}")
        if 'llm_confidence' in v:
            print(f"      置信度: LLM={v['llm_confidence']:.2%}, 计算={v['calculated_confidence']:.2%}")
        else:
            print(f"      置信度: {v.get('confidence', 0):.2%}")
        print(f"      原因: {v.get('reason', 'N/A')[:100]}...")
else:
    print("✅ 无违规项")
    print()

print("="*70)
print("📚 检索到的相关条文 (Top-5):")
print("="*70)
rules = result.get('retrieved_rules', [])[:5]
for i, rule in enumerate(rules, 1):
    print(f"\n[{i}] {rule.get('clause_id', 'N/A')} | 相似度: {rule.get('score', 0):.4f}")
    print(f"    来源: {rule.get('source_file', 'N/A')}")
    print(f"    条文: {rule.get('clause_text', 'N/A')[:100]}...")

## 🎯 3. 示例文案快速测试

In [ ]:
# 示例1: 夸大收益（违规）
example1 = "本保险产品保本保收益，年化收益率保证8%，零风险，稳赚不赔！"

result1 = audit_marketing_text(example1, kb, client, top_k=6)
print(f"是否合规: {result1['is_compliant']}")
print(f"置信度: {result1['overall_confidence']:.2%}")
print(f"总结: {result1['summary']}")

In [ ]:
# 示例2: 合规文案
example2 = "本产品过往业绩不代表未来表现，具体以合同条款为准。投资有风险，投保需谨慎。"

result2 = audit_marketing_text(example2, kb, client, top_k=6)
print(f"是否合规: {result2['is_compliant']}")
print(f"置信度: {result2['overall_confidence']:.2%}")
print(f"总结: {result2['summary']}")

## 📊 4. 批量测试与可视化

In [ ]:
# 批量测试多个文案
test_cases = [
    "本保险保证年化收益8%，稳赚不赔。",
    "限时抢购，购买后稳赚不赔，现在下单即可翻倍。",
    "本产品过往业绩不代表未来表现，具体以合同条款为准。",
    "请您仔细阅读保险条款，重点关注责任免除和犹豫期。",
    "VIP专享理财型保险，保证年化收益率10%，比银行存款高5倍！",
    "知名影星实名推荐，本保险产品收益稳定，跟着明星买准没错！",
]

# 执行审核
results = []
for text in test_cases:
    result = audit_marketing_text(text, kb, client, top_k=6)
    results.append({
        'text': text,
        'is_compliant': result['is_compliant'],
        'confidence': result['overall_confidence'],
        'violations_count': len(result['violations']),
    })

# 创建DataFrame
df = pd.DataFrame(results)
display(df)

# 统计
compliant_count = sum(1 for r in results if r['is_compliant'] == 'yes')
non_compliant_count = len(results) - compliant_count
accuracy = (df['is_compliant'] == 'yes').mean()

print(f"\n📊 统计结果:")
print(f"   - 总测试数: {len(results)}")
print(f"   - 合规数量: {compliant_count}")
print(f"   - 不合规数量: {non_compliant_count}")
print(f"   - 平均置信度: {df['confidence'].mean():.2%}")

In [ ]:
# 可视化：合规性分布
import matplotlib.pyplot as plt

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 创建图表
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 饼图：合规 vs 不合规
compliance_counts = df['is_compliant'].value_counts()
axes[0].pie(compliance_counts.values, labels=compliance_counts.index, autopct='%1.1f%%')
axes[0].set_title('合规性分布')

# 条形图：置信度分布
colors = ['green' if c == 'yes' else 'red' for c in df['is_compliant']]
axes[1].bar(range(len(df)), df['confidence'], color=colors)
axes[1].set_xlabel('测试用例')
axes[1].set_ylabel('置信度')
axes[1].set_title('各用例置信度')
axes[1].set_xticks(range(len(df)))
axes[1].set_xticklabels([f'用例{i+1}' for i in range(len(df))], rotation=45)
plt.tight_layout()

plt.show()

print("\n📊 图表已生成！")

## 🔧 5. 自定义测试

In [ ]:
# 📝 在这里输入你的自定义文案
custom_text = """  # 修改这里

# 执行审核
custom_result = audit_marketing_text(custom_text, kb, client, top_k=6)

# 显示结果
print(f"是否合规: {custom_result['is_compliant']}")
print(f"置信度: {custom_result['overall_confidence']:.2%}")
print(f"\n总结: {custom_result['summary']}")

if custom_result['violations']:
    print("\n⚠️ 违规项:")
    for v in custom_result['violations']:
        print(f"  - {v['type']}: {v['reason'][:80]}...")

## 💡 使用说明

1. **运行顺序**：按顺序执行所有单元格（Cell → Run All）
2. **修改文案**：在「2. 文本审核 Demo」单元格中修改 `marketing_text`
3. **切换模式**：修改 `use_enhanced` 和 `use_hybrid` 变量
4. **自定义测试**：在「5. 自定义测试」单元格中输入你的文案

## 🎯 核心功能

| 功能 | 单元格 |
|------|--------|
| 资源加载 | 1. 加载资源 |
| 文本审核 | 2. 文本审核 Demo |
| 示例测试 | 3. 示例文案快速测试 |
| 批量测试 | 4. 批量测试与可视化 |
| 自定义 | 5. 自定义测试 |